# Exercise 02: Schema Design & Constraints

One of the most important things you can do to protect data quality in a warehouse is to encode your **business rules directly into the schema**.

In this exercise you'll design a schema for a company's HR system, applying six different types of constraints to enforce data quality at the database level.

---

### The six constraints you'll use

| Constraint | What it does | Example |
|---|---|---|
| `PRIMARY KEY` | Uniquely identifies each row. Cannot be NULL or duplicate. | `employee_id INTEGER PRIMARY KEY` |
| `FOREIGN KEY` | Links a column to a primary key in another table. Prevents orphaned rows. | `REFERENCES departments(department_id)` |
| `UNIQUE` | No two rows can have the same value. Can be NULL. | `email VARCHAR UNIQUE` |
| `NOT NULL` | The column must always have a value. | `name VARCHAR NOT NULL` |
| `CHECK` | Value must pass a condition. | `salary CHECK (salary >= 0)` |
| `DEFAULT` | Value used when none is provided on INSERT. | `status VARCHAR DEFAULT 'active'` |

---

### The schema you'll build

```
Schema: company
  departments   — one row per department
  employees     — one row per employee, linked to a department
```

## Setup

In [1]:
import duckdb

con = duckdb.connect()
print('Connected! DuckDB version:', duckdb.__version__)

Connected! DuckDB version: 1.5.2


---
## Task 1: Create a Schema

A **schema** is a namespace that groups related tables together.
In production warehouses you'll commonly see schemas like `raw`, `staging`, `marts`, `finance`, `hr`, etc.

Create a schema called `company`.

```sql
CREATE SCHEMA schema_name;
```

To create a table inside a schema, prefix the table name: `company.departments`.

In [2]:
# ✏️ Create the 'company' schema
con.execute('''
    CREATE SCHEMA company
''')

In [3]:
# Check your work — 'company' should appear in the list
con.execute('''
    SELECT schema_name
    FROM information_schema.schemata
    ORDER BY schema_name
''').df()

,schema_name
0,company
1,information_schema
2,main
3,main
4,main
5,pg_catalog


---
## Task 2: Create the `departments` table

Design the `company.departments` table with the following columns and constraints:

| Column | Type | Constraints | Notes |
|---|---|---|---|
| `department_id` | `INTEGER` | `PRIMARY KEY` | Unique identifier |
| `name` | `VARCHAR` | `NOT NULL`, `UNIQUE` | No two departments can share a name |
| `budget` | `DECIMAL(15, 2)` | `NOT NULL`, `CHECK (budget > 0)` | Must be a positive number |

In [4]:
con.execute('''
        CREATE TABLE company.departments (
        department_id INTEGER PRIMARY KEY,
        name          VARCHAR        NOT NULL UNIQUE,
        budget        DECIMAL(15, 2) NOT NULL CHECK (budget > 0)
    )
''')

In [5]:
# Check your table structure
con.execute('''
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_schema = 'company' AND table_name = 'departments'
    ORDER BY ordinal_position
''').df()

,column_name,data_type,is_nullable
0,department_id,INTEGER,NO
1,name,VARCHAR,NO
2,budget,"DECIMAL(15,2)",NO


---
## Task 3: Create the `employees` table

Design the `company.employees` table. This table uses all six constraint types.

| Column | Type | Constraints | Notes |
|---|---|---|---|
| `employee_id` | `INTEGER` | `PRIMARY KEY` | Unique identifier |
| `first_name` | `VARCHAR` | `NOT NULL` | |
| `last_name` | `VARCHAR` | `NOT NULL` | |
| `email` | `VARCHAR` | `NOT NULL`, `UNIQUE` | No duplicate email addresses |
| `department_id` | `INTEGER` | `REFERENCES company.departments(department_id)` | Foreign key |
| `salary` | `DECIMAL(10, 2)` | `NOT NULL`, `CHECK (salary >= 0)` | Must be non-negative |
| `hire_date` | `DATE` | `NOT NULL`, `DEFAULT CURRENT_DATE` | Defaults to today |
| `employment_status` | `VARCHAR` | `NOT NULL`, `DEFAULT 'active'`, `CHECK (employment_status IN ('active', 'inactive', 'on-leave'))` | Only three allowed values |

In [6]:
con.execute('''
    CREATE TABLE company.employees (
        employee_id       INTEGER        PRIMARY KEY,
        first_name        VARCHAR        NOT NULL,
        last_name         VARCHAR        NOT NULL,
        email             VARCHAR        NOT NULL UNIQUE,
        department_id     INTEGER        REFERENCES company.departments(department_id),
        salary            DECIMAL(10, 2) NOT NULL CHECK (salary >= 0),
        hire_date         DATE           NOT NULL DEFAULT CURRENT_DATE,
        employment_status VARCHAR        NOT NULL DEFAULT 'active'
                                                  CHECK (employment_status IN ('active', 'inactive', 'on-leave'))
    )
''')

In [7]:
# Check your table structure
con.execute('''
    SELECT column_name, data_type, is_nullable, column_default
    FROM information_schema.columns
    WHERE table_schema = 'company' AND table_name = 'employees'
    ORDER BY ordinal_position
''').df()

,column_name,data_type,is_nullable,column_default
0,employee_id,INTEGER,NO,NaN
1,first_name,VARCHAR,NO,NaN
2,last_name,VARCHAR,NO,NaN
3,email,VARCHAR,NO,NaN
4,department_id,INTEGER,YES,NaN
5,salary,"DECIMAL(10,2)",NO,NaN
6,hire_date,DATE,NO,CURRENT_DATE
7,employment_status,VARCHAR,NO,'active'


---
## Task 4: Insert Data

Populate both tables with some sample data.
Departments must be inserted before employees (because of the foreign key).

In [8]:
# Insert departments first
con.execute('''
    INSERT INTO company.departments (department_id, name, budget) VALUES
        (1, 'Engineering',  2500000.00),
        (2, 'Marketing',     800000.00),
        (3, 'Operations',   1200000.00),
        (4, 'Data',          950000.00)
''')

con.execute('SELECT * FROM company.departments').df()

,department_id,name,budget
0,1,Engineering,2500000.0
1,2,Marketing,800000.0
2,3,Operations,1200000.0
3,4,Data,950000.0


In [9]:
# ✏️ Insert at least 5 employees into company.employees
# Notice: you don't need to supply hire_date or employment_status — the DEFAULTs will fill them in
con.execute('''
    INSERT INTO company.employees (employee_id, first_name, last_name, email, department_id, salary)
    VALUES
        (1, 'Alice',   'Smith',    'alice@company.com',   1, 95000.00),
        (2, 'Bob',     'Johnson',  'bob@company.com',     2, 82000.00),
        (3, 'Carol',   'Williams', 'carol@company.com',   1, 110000.00),
        (4, 'Dave',    'Brown',    'dave@company.com',    3, 74000.00),
        (5, 'Eve',     'Garcia',   'eve@company.com',     2, 91500.00),
        (6, 'Frank',   'Martinez', 'frank@company.com',   3, 67000.00)
''')

con.execute('SELECT * FROM company.employees').df()

,employee_id,first_name,last_name,email,department_id,salary,hire_date,employment_status
0,1,Alice,Smith,alice@company.com,1,95000.0,2026-04-24,active
1,2,Bob,Johnson,bob@company.com,2,82000.0,2026-04-24,active
2,3,Carol,Williams,carol@company.com,1,110000.0,2026-04-24,active
3,4,Dave,Brown,dave@company.com,3,74000.0,2026-04-24,active
4,5,Eve,Garcia,eve@company.com,2,91500.0,2026-04-24,active
5,6,Frank,Martinez,frank@company.com,3,67000.0,2026-04-24,active


---
## Task 5: Test Your Constraints

Each cell below tries to INSERT data that should be rejected.
Run each one and observe the error message. This is how constraints protect your data!

> These cells are **expected to fail**. Read the error message — it tells you exactly which constraint was violated.

In [10]:
# Test PRIMARY KEY — inserting a duplicate employee_id should fail
try:
    con.execute('''
        INSERT INTO company.employees (employee_id, first_name, last_name, email, department_id, salary)
        VALUES (1, 'Duplicate', 'Person', 'dup@company.com', 1, 50000)
    ''')
    print('❌ Insert succeeded — PRIMARY KEY not enforced!')
except Exception as e:
    print('✅ PRIMARY KEY violation caught:', e)

✅ PRIMARY KEY violation caught: Constraint Error: Duplicate key "employee_id: 1" violates primary key constraint.


In [11]:
# Test CHECK constraint — a negative salary should fail
try:
    con.execute('''
        INSERT INTO company.employees (employee_id, first_name, last_name, email, department_id, salary)
        VALUES (999, 'Bad', 'Salary', 'bad@company.com', 1, -100)
    ''')
    print('❌ Insert succeeded — CHECK constraint not enforced!')
except Exception as e:
    print('✅ CHECK violation caught:', e)

✅ CHECK violation caught: Constraint Error: CHECK constraint failed on table employees with expression CHECK((salary >= 0))


In [12]:
# Test FOREIGN KEY — referencing a department_id that doesn't exist should fail
try:
    con.execute('''
        INSERT INTO company.employees (employee_id, first_name, last_name, email, department_id, salary)
        VALUES (998, 'Lost', 'Employee', 'lost@company.com', 999, 60000)
    ''')
    print('❌ Insert succeeded — FOREIGN KEY not enforced!')
except Exception as e:
    print('✅ FOREIGN KEY violation caught:', e)

✅ FOREIGN KEY violation caught: Constraint Error: Violates foreign key constraint because key "department_id: 999" does not exist in the referenced table


In [13]:
# Test UNIQUE — duplicate email address should fail
try:
    con.execute('''
        INSERT INTO company.employees (employee_id, first_name, last_name, email, department_id, salary)
        VALUES (997, 'Same', 'Email', 'alice@company.com', 1, 60000)
    ''')
    print('❌ Insert succeeded — UNIQUE constraint not enforced!')
except Exception as e:
    print('✅ UNIQUE violation caught:', e)

✅ UNIQUE violation caught: Constraint Error: Duplicate key "email: alice@company.com" violates unique constraint.


In [14]:
# Test CHECK on employment_status — an invalid status value should fail
try:
    con.execute('''
        INSERT INTO company.employees (employee_id, first_name, last_name, email, department_id, salary, employment_status)
        VALUES (996, 'Bad', 'Status', 'status@company.com', 1, 60000, 'fired')
    ''')
    print('❌ Insert succeeded — CHECK constraint on employment_status not enforced!')
except Exception as e:
    print('✅ CHECK violation caught:', e)

✅ CHECK violation caught: Constraint Error: CHECK constraint failed on table employees with expression CHECK((employment_status IN ('active', 'inactive', 'on-leave')))


---
## Task 6: Explore Constraints in the Information Schema

Just as you can query `information_schema.columns` to see column definitions, you can query DuckDB's built-in functions to inspect constraints.

In [15]:
# View all constraints across our schema
con.execute('''
    SELECT
        schema_name,
        table_name,
        constraint_index,
        constraint_type,
        constraint_text
    FROM duckdb_constraints()
    WHERE schema_name = 'company'
    ORDER BY table_name, constraint_index
''').df()

,schema_name,table_name,constraint_index,constraint_type,constraint_text
0,company,departments,0,PRIMARY KEY,PRIMARY KEY(department_id)
1,company,departments,1,NOT NULL,NOT NULL
2,company,departments,2,UNIQUE,"UNIQUE(""name"")"
3,company,departments,3,NOT NULL,NOT NULL
4,company,departments,4,CHECK,CHECK((budget > 0))
5,company,departments,5,NOT NULL,NOT NULL
6,company,employees,6,PRIMARY KEY,PRIMARY KEY(employee_id)
7,company,employees,7,NOT NULL,NOT NULL
8,company,employees,8,NOT NULL,NOT NULL
9,company,employees,9,NOT NULL,NOT NULL


In [16]:
# View the final state of your employees table
con.execute('''
    SELECT
        e.employee_id,
        e.first_name,
        e.last_name,
        e.email,
        d.name AS department,
        e.salary,
        e.hire_date,
        e.employment_status
    FROM company.employees e
    JOIN company.departments d ON e.department_id = d.department_id
    ORDER BY e.employee_id
''').df()

,employee_id,first_name,last_name,email,department,salary,hire_date,employment_status
0,1,Alice,Smith,alice@company.com,Engineering,95000.0,2026-04-24,active
1,2,Bob,Johnson,bob@company.com,Marketing,82000.0,2026-04-24,active
2,3,Carol,Williams,carol@company.com,Engineering,110000.0,2026-04-24,active
3,4,Dave,Brown,dave@company.com,Operations,74000.0,2026-04-24,active
4,5,Eve,Garcia,eve@company.com,Marketing,91500.0,2026-04-24,active
5,6,Frank,Martinez,frank@company.com,Operations,67000.0,2026-04-24,active


---
## Summary

| Constraint | Why it matters |
|---|---|
| `PRIMARY KEY` | Every row is uniquely addressable — essential for JOINs |
| `FOREIGN KEY` | Prevents orphaned rows — referential integrity |
| `UNIQUE` | Prevents duplicates on business keys (emails, codes) |
| `NOT NULL` | Ensures critical fields always have values |
| `CHECK` | Encodes business rules directly in the schema |
| `DEFAULT` | Makes inserts simpler and reduces NULL surprises |

These constraints are your **first line of defence** against bad data entering your warehouse.